# Paper §3.2: Thermal convection, Rayleigh–Bénard

This notebook reproduces the Rayleigh–Bénard figure of Section 3.2 of the paper *GeLB: A Domain-Specific Language and Compiler for Lattice Boltzmann Simulations* (Chirila & Lohmann, submitted to SoftwareX, 2026), with the GeLB programs the paper used ([`examples/softwarex-2026-paper/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/softwarex-2026-paper), release v1.0.0).

A fluid layer heated from below and cooled from above, at Rayleigh number $\mathrm{Ra} = 10^4$ and Prandtl number 0.71: two coupled lattices, D2Q9 for the flow and D2Q5 for the temperature, with Boussinesq buoyancy. Above the critical Rayleigh number (about 1708) a small perturbation grows into steady convection rolls; in this 128 × 32 box they settle into six cells.

The paper's figure, and its listing of the collision step with Strang-split buoyancy, come from the MRT program. The paper's appendix lists the BGK program in full: it differs only in its collision step (and the constants that go with it), is seeded the same way, and convects at the same Ra. Both runs take 400 000 steps.

The notebook runs on a free Colab CPU runtime. Expected runtime: **3–6 minutes** (100 s of simulation on a laptop with a Ryzen 9 7945HX).

## 1. Install the compilers' runtime dependencies

The Fortran code that `gelbc` emits needs `gfortran` and the NetCDF-Fortran library, which writes the gauge output.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq gfortran libnetcdff-dev > /dev/null
!gfortran --version | head -1

## 2. Install GeLB

GeLB 1.0.0, the release described in the paper, installs from its tag on GitHub (about 30 s). The CPU runs use the Fortran backend (`-b cpu_fortran_serial`): a pip installation has no Highway library, which the default `cpu_highway` backend needs (see the [README](https://github.com/dchirila/gelb#installation)). The backends compute the same results; the paper's timings used `cpu_highway`.

In [ ]:
!pip install -q "git+https://github.com/dchirila/gelb.git@v1.0.0"

# gelb-run starts the compiler and its progress monitor through a Python
# interpreter that can import gelb: name this notebook's own.
import os, sys
os.environ["GELB_PYTHON"] = sys.executable

!python -c "import importlib.metadata as m; print('gelb', m.version('gelb'))"

GeLB source is shown with syntax highlighting by the Pygments lexer that the GeLB manuals use:

In [ ]:
!curl -sSfL -o gelb_lexer.py https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/docs/_ext/gelb_lexer.py

from gelb_lexer import GelbLexer
from pygments import highlight
from pygments.formatters import HtmlFormatter
from IPython.display import HTML, display

def show_gelb(path, max_height=None):
    """Display a .gelb file with syntax highlighting and line numbers."""
    with open(path) as f:
        html = highlight(f.read(), GelbLexer(),
                         HtmlFormatter(noclasses=True, style="friendly", linenos="inline",
                                       prestyles="font-family: monospace; font-size: 13px; line-height: 125%;"))
    box = f"max-height: {max_height}px; overflow: auto;" if max_height else ""
    display(HTML(f'<div style="{box}">{html}</div>'))

## 3. Get the paper's programs

In [ ]:
BASE = "https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/examples/softwarex-2026-paper"
for name in ["rayleigh_benard_2d_mrt.gelb", "rayleigh_benard_2d_bgk.gelb"]:
    !curl -sSfL -o {name} {BASE}/{name}
!ls -l *.gelb

The BGK program, as listed in the paper's appendix:

In [ ]:
show_gelb('rayleigh_benard_2d_bgk.gelb', max_height=500)

The MRT program, whose collision step is the paper's listing in Section 3.2:

In [ ]:
show_gelb('rayleigh_benard_2d_mrt.gelb', max_height=500)

## 4. Run the MRT program (the paper's figure)

The gauge writes the temperature and velocity every 2 000 steps. The initial perturbation includes noise seeded from the clock, so the rolls can sit a little to the left or right of the paper's; `-s 42` (any integer) fixes the seed.

In [ ]:
!gelb-run -b cpu_fortran_serial -n 400000 -w 2000 rayleigh_benard_2d_mrt.gelb

In [ ]:
import netCDF4
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# The paper's palette (Okabe-Ito): blue = read / cold, vermillion = write / warm.
BLUE, VERMILLION, GREEN, GRAY, INK = "#0072B2", "#D55E00", "#009E73", "#878787", "#333333"
DIVERGING = LinearSegmentedColormap.from_list("gelb_div", [BLUE, "#f4f3f1", VERMILLION])

def last_record(nc, *names):
    """The last gauge record of each variable, with non-fluid cells as NaN."""
    with netCDF4.Dataset(nc) as ds:
        return [np.ma.filled(ds.variables[n][-1], np.nan) for n in names]

In [ ]:
def rb_snapshot(nc, ux_name, uy_name, title):
    T, ux, uy = last_record(nc, "T", ux_name, uy_name)
    ny, nx = T.shape
    fig, ax = plt.subplots(figsize=(10, 3))
    im = ax.imshow(T, origin="lower", cmap=DIVERGING, vmin=0.0, vmax=1.0,
                   extent=(0, nx, 0, ny), interpolation="bilinear")
    ax.streamplot(np.arange(nx), np.arange(ny), ux, uy, color="#555555",
                  linewidth=0.6, density=(1.6, 0.5), arrowsize=0.6)
    ax.set_xlim(0, nx); ax.set_ylim(0, ny)
    ax.set_xlabel("$x$ (lattice units)"); ax.set_ylabel("$y$"); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="temperature $T$")
    plt.tight_layout(); plt.show()

def rb_growth(nc, uy_name, label, color):
    with netCDF4.Dataset(nc) as ds:
        t = ds.variables["time"][:]
        uy = np.ma.filled(ds.variables[uy_name][:], np.nan)
    plt.semilogy(t[1:], np.nanmax(np.abs(uy[1:]), axis=(1, 2)), "-", color=color, label=label)

rb_snapshot("rayleigh_benard_2d_mrt.nc", "uX", "uY", "MRT, Ra = 1e4, step 400 000 (the paper's figure)")

## 5. Run the BGK program

In [ ]:
!gelb-run -b cpu_fortran_serial -n 400000 -w 2000 rayleigh_benard_2d_bgk.gelb

In [ ]:
rb_snapshot("rayleigh_benard_2d_bgk.nc", "ux", "uy", "BGK, Ra = 1e4, step 400 000")

## 6. Onset and saturation

The largest vertical velocity in the box, every 2 000 steps: after a short transient, the unstable mode grows exponentially (a straight line on this log scale) and levels off by about step 40 000, once the rolls are steady.

In [ ]:
plt.figure(figsize=(6, 3.5))
rb_growth("rayleigh_benard_2d_mrt.nc", "uY", "MRT", VERMILLION)
rb_growth("rayleigh_benard_2d_bgk.nc", "uy", "BGK", BLUE)
plt.xlabel("time step"); plt.ylabel("max |u_y|")
plt.grid(True, which="both", alpha=0.3); plt.legend(frameon=False)
plt.tight_layout(); plt.show()

To try another Rayleigh number, add `-D RA=...` to the `gelb-run` command: at `-D RA=1500`, below onset, the perturbation decays instead of growing. The paper's critical Rayleigh number, $\mathrm{Ra}_c \approx 1707.5$ against 1707.76 from linear stability theory, comes from runs of ten million steps per Rayleigh number just below and above onset; they are meant for a local checkout (`scripts/find_ra_critical.py` in the [repository](https://github.com/dchirila/gelb)).

## Where to go from here

- The programs and the commands that produced the paper's figures: [`examples/softwarex-2026-paper/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/softwarex-2026-paper).
- The [GeLB User Tutorial](https://github.com/dchirila/gelb/blob/v1.0.0/docs/pdf/GeLB_User_Tutorial.pdf) (PDF) and the [repository](https://github.com/dchirila/gelb).
- Release 1.0.0 is archived as [doi:10.5281/zenodo.23592922](https://doi.org/10.5281/zenodo.23592922); the [README](https://github.com/dchirila/gelb#using-gelb-in-your-work) says how to cite GeLB.